In [1]:
import pandas as pd
import numpy as np
import pymssql
import sqlalchemy as db
import calendar
from datetime import datetime, timedelta
from sklearn.metrics import mean_absolute_error
import warnings
from copy import copy, deepcopy

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# DS Claim Level Pred Data

In [2]:
def db_connection():
    con = pymssql.connect(user='',password = '',host='',database='',autocommit = True)
    cursor = con.cursor()

    return con, cursor

def db_disconnect(conn, cursor):
    cursor.close()
    conn.close()

In [3]:
def data_freeze():
    
    del_query = """
    query
    """

    query = """

    query

    """
    con, cursor = clnjdb7()

    cursor.execute(del_query)
    cursor.execute(query)
    con.commit()

    db_disconnect(con, cursor)

    print("Data Freezed Successfully...!")

    return 0


In [4]:
def pred_data():
    query = """query
    """
    
    con, cursor = clnjdb7()
    
    df = pd.read_sql(query, con)
    db_disconnect(con, cursor)
    return df

In [5]:
df = pred_data()

In [6]:
# df = pd.read_csv("pred_fin_new.csv")

In [8]:
df.shape[0]

15144

In [9]:
df.PRACTICE_CODE.unique()

array([100, 111])

In [10]:
df.EXPECTED_APPROVED_AMT.sum(), df.EXPECTED_PAID_AMT.sum()

(np.float64(442961.83999999997), np.float64(331450.615))

In [11]:
df['EXPECTED_PAYMENT_DATE'] = pd.to_datetime(df['EXPECTED_PAYMENT_DATE'])

In [12]:
df.drop(columns={'ID','DELETED','CREATED_DATE','CREATED_BY','MODIFIED_DATE','MODIFIED_BY','ENTERED_FROM'}, inplace=True)

In [13]:
df_new = df.loc[df.groupby(['CLAIM_NO','PROCEDURE_CODE','INSPAYER_ID'])['PROCESS_DATE'].idxmin()].reset_index(drop=True)

In [14]:
df_new.shape

(15144, 70)

# Payments Data

In [15]:
def pay_data():
    query = """query"""
    
    con, cursor = clnjdb7()
    
    payments = pd.read_sql(query, con)
    db_disconnect(con, cursor)
    return payments

In [16]:
payments = pay_data()

In [17]:
payments.columns = payments.columns.str.upper()
payments['DATE_ENTRY'] = pd.to_datetime(payments['DATE_ENTRY'])
payments.rename(columns={'PAID_PROC_CODE':'PROCEDURE_CODE'}, inplace=True)

# Actual and Pred Merging


In [18]:
payment_fin = deepcopy(payments)

In [19]:
payment_fin.rename(columns={'DESCRIPTION': 'REJECTION_REASON'},inplace=True)

In [20]:
payment_fin[['REJECT_TYPE', 'ERA_REJECTION_CATEGORY_CODE', 'ERA_CATEGORY_CODE', 'ERA_ADJUSTMENT_CODE', 'REJECTION_REASON','ENTERED_FROM']] = \
payment_fin[['REJECT_TYPE', 'ERA_REJECTION_CATEGORY_CODE', 'ERA_CATEGORY_CODE', 'ERA_ADJUSTMENT_CODE', 'REJECTION_REASON','ENTERED_FROM']].fillna('')

In [21]:
payment_fin = payment_fin.groupby(['CLAIM_NO','INSURANCE_ID','DATE_ENTRY','PROCEDURE_CODE','REJECT_TYPE',
        'ERA_REJECTION_CATEGORY_CODE','ERA_CATEGORY_CODE','ERA_ADJUSTMENT_CODE','REJECTION_REASON','ENTERED_FROM'])[['AMOUNT_APPROVED','AMOUNT_PAID','AMOUNT_ADJUSTED','REJECT_AMOUNT']].sum().reset_index()

In [22]:
df_new['CLAIM_NO'] = df_new['CLAIM_NO'].astype(int)
df_new['INSURANCE_ID'] = df_new['INSURANCE_ID'].astype(int)
# df_new['PROCEDURE_CODE'] = df_new['PROCEDURE_CODE'].astype(str)

payment_fin['CLAIM_NO'] = payment_fin['CLAIM_NO'].astype(int)
payment_fin['INSURANCE_ID'] = payment_fin['INSURANCE_ID'].astype(int)
# payment_fin['PROCEDURE_CODE'] = payment_fin['PROCEDURE_CODE'].astype(str)

df_new['PROCEDURE_CODE'] = df_new['PROCEDURE_CODE'].astype(str).str.strip()
payment_fin['PROCEDURE_CODE'] = payment_fin['PROCEDURE_CODE'].astype(str).str.strip()

In [23]:
merg = df_new.merge(payment_fin, on=['CLAIM_NO','INSURANCE_ID','PROCEDURE_CODE'], how='left')

In [24]:
merg['EXPECTED_PAYMENT_DATE'] = pd.to_datetime(merg['EXPECTED_PAYMENT_DATE'])
merg['DATE_ENTRY'] = pd.to_datetime(merg['DATE_ENTRY'])

In [25]:
merg_new = merg.sort_values(by=['CLAIM_NO','PROCESS_DATE','DATE_ENTRY'])

In [26]:
# Filter rows where DATE_ENTRY is greater than PROCESS_DATE or DATE_ENTRY is null
merg_new_fin = merg_new[
    (merg_new['DATE_ENTRY'].notna() & (merg_new['DATE_ENTRY'] > merg_new['PROCESS_DATE'])) |
    (merg_new['DATE_ENTRY'].isna())
]

In [27]:
merg_new_fin.shape[0]

17453

In [28]:
merg_new_fin['PROCESS_DATE'] = pd.to_datetime(merg_new_fin['PROCESS_DATE'], errors='coerce')
# merg_new_fin[merg_new_fin['PROCESS_DATE'].dt.year==2025]['CLAIM_NO'].nunique()

In [29]:
merg['REJECT_TYPE'].fillna('', inplace=True)

In [30]:
#For first payment date after submission date if AMOUNT_APPROVED is 0 its mean the claim was resubmitted
# Convert Date_Entry to datetime for proper comparison
merg_new_fin['DATE_ENTRY'] = pd.to_datetime(merg_new_fin['DATE_ENTRY'])

# Group by Claim_No and Date_Entry to calculate the sum of Amount_Approved
grouped = merg_new_fin.groupby(['CLAIM_NO', 'DATE_ENTRY'])['AMOUNT_APPROVED'].sum().reset_index()

# Find the minimum Date_Entry for each Claim_No and its corresponding sum of Amount_Approved
min_date_amount = grouped.loc[grouped.groupby('CLAIM_NO')['DATE_ENTRY'].idxmin()]

# Create a mapping based on the condition
claim_mapping = min_date_amount.groupby('CLAIM_NO').apply(
    lambda x: 1 if x['AMOUNT_APPROVED'].sum() < 1 else 0
).to_dict()

# Add a new column 'ReSubmitted' to the original DataFrame based on the Claim_No
merg_new_fin['RESUBMITTED'] = merg_new_fin['CLAIM_NO'].map(claim_mapping)

In [31]:
merg_new_fin_paid = merg_new_fin[~(merg_new_fin['DATE_ENTRY'].isna())]
merg_new_fin_not_paid = merg_new_fin[merg_new_fin['DATE_ENTRY'].isna()]

In [32]:
# merg_new_fin

In [33]:

# Step 1: Group by Claim_No and Date_Entry and calculate the sum of Amount_Approved
grouped_sums = merg_new_fin_paid.groupby(['CLAIM_NO', 'DATE_ENTRY'])['AMOUNT_APPROVED'].sum()

# Step 2: Filter groups where the sum is greater than 1
valid_groups = grouped_sums[grouped_sums > 1].index  # Get valid (Claim_No, Date_Entry) pairs

In [34]:
# grouped_sums

In [35]:
# Step 3: Filter the DataFrame using the valid groups
merg_new_fin_upd = merg_new_fin_paid[
    merg_new_fin_paid.set_index(['CLAIM_NO', 'DATE_ENTRY']).index.isin(valid_groups)
].reset_index(drop=True)

In [36]:
merg_new_fin_upd.shape[0], merg_new_fin.shape[0]

(11673, 17453)

In [37]:
# merg_new_fin

In [38]:
# Apply row numbers partitioned by Claim_No and Process_Date, ordered by Date_Entry
merg_new_fin_upd["Row_Number"] = (
    merg_new_fin_upd.sort_values(["CLAIM_NO", "DATE_ENTRY"])  # Sort by Claim_No and Date_Entry
    .groupby("CLAIM_NO")["DATE_ENTRY"]  # Group by Claim_No and focus on Date_Entry
    .rank(method="dense")  # Assign dense rank for Date_Entry
    .astype(int)  # Convert to integer for clean row numbers
)

In [39]:
merg_new_row = merg_new_fin_upd[merg_new_fin_upd['Row_Number']==1]

In [40]:
final_dfff = pd.concat([merg_new_row,merg_new_fin_not_paid]).reset_index(drop=True)

In [41]:
# Step 1: Create a DataFrame with counts
grouped_counts = (
    final_dfff.groupby(['CLAIM_NO', 'DATE_ENTRY', 'PROCEDURE_CODE'])
    .size()
    .reset_index(name='Count')
)

# Step 2: Filter for groups where Count > 1
filtered_groups = grouped_counts.query('Count > 1')

# Step 3: Merge with the original DataFrame to identify matching rows
merged_with_counts = final_dfff.merge(
    filtered_groups,
    on=['CLAIM_NO', 'DATE_ENTRY', 'PROCEDURE_CODE'],
    how='left'
)

# Step 4: Remove rows where Count > 1 and AMOUNT_APPROVED = 0
filtered_merg_new_row = merged_with_counts[
    ~((merged_with_counts['Count'] > 1) & (merged_with_counts['AMOUNT_APPROVED'] == 0))
]

# Step 5: Drop the Count column (optional)
filtered_merg_new_row = filtered_merg_new_row.drop(columns=['Count'])

In [42]:
duplicates_summary = (
    filtered_merg_new_row.groupby(['CLAIM_NO', 'DATE_ENTRY', 'PROCEDURE_CODE'])
    .size()
    .reset_index(name='Count')
    .query('Count > 1')  # Keep only those with more than one occurrence
)

In [43]:
duplicates_summary.shape[0]

25

In [44]:
duplicates_summary = duplicates_summary.sort_values(by='Count', ascending=False).reset_index(drop=True)

In [45]:
duplicates_summary.head(2)

,CLAIM_NO,DATE_ENTRY,PROCEDURE_CODE,Count
0,9283669335,2025-06-25,3008F,4
1,9283669335,2025-06-25,93000,3


In [47]:
filtered_merg_new_row.AMOUNT_APPROVED.sum(), filtered_merg_new_row.EXPECTED_APPROVED_AMT.sum(),df_new.EXPECTED_APPROVED_AMT.sum()

(np.float64(394533.84), np.float64(438070.06999999995), np.float64(442961.84))

In [48]:
filtered_merg_new_row['PROCESS_DATE'] = pd.to_datetime(filtered_merg_new_row['PROCESS_DATE'], errors='coerce')
filtered_merg_new_row.rename(columns={'DOE':'CHARGE_ENTRY_DATE','PROCESS_DATE':'SUBMISSION_DATE',
        'AMOUNT':'CHARGE_AMOUNT','DATE_ENTRY':'PAYMENT_DATE'}, inplace=True)

In [49]:
filtered_merg_new_row['AMOUNT_APPROVED'] = filtered_merg_new_row['AMOUNT_APPROVED'].round(2)
filtered_merg_new_row['AMOUNT_PAID'] = filtered_merg_new_row['AMOUNT_PAID'].round(2)
filtered_merg_new_row['REJECT_AMOUNT'] = filtered_merg_new_row['REJECT_AMOUNT'].round(2)
filtered_merg_new_row['AMOUNT_ADJUSTED'] = filtered_merg_new_row['AMOUNT_ADJUSTED'].round(2)
# filtered_merg_new_row['RESUBMITTED'] = filtered_merg_new_row['RESUBMITTED'].astype(int)

In [51]:
filtered_merg_new_row.drop(columns={'Row_Number'}, inplace=True)
filtered_merg_new_row['REJECT_TYPE'].fillna('',inplace=True)

In [52]:
filtered_merg_new_row.rename(columns={'DOE':'CHARGE_ENTRY_DATE','PROCESS_DATE':'SUBMISSION_DATE',
                                      'AMOUNT':'CHARGE_AMOUNT', 'DATE_ENTRY':'PAYMENT_DATE'}, inplace=True)

In [53]:
filtered_merg_new_row['SUBMISSION_DATE'] = pd.to_datetime(filtered_merg_new_row['SUBMISSION_DATE'], errors='coerce')

In [54]:
filtered_merg_new_row.shape[0], df.shape[0]

(14184, 15144)

In [55]:
filtered_merg_new_row.EXPECTED_PAID_AMT.sum(), filtered_merg_new_row.AMOUNT_PAID.sum(), filtered_merg_new_row.EXPECTED_APPROVED_AMT.sum(), filtered_merg_new_row.AMOUNT_APPROVED.sum()

(np.float64(328155.455),
 np.float64(294501.26),
 np.float64(438070.06999999995),
 np.float64(394533.84))

In [56]:
filtered_merg_new_row.CLAIM_NO.nunique()

4438

In [57]:
# filtered_merg_new_row.SUBMISSION_DATE.unique()

In [58]:
filtered_merg_new_row.shape[0]

14184

In [60]:
def delete_data():
    query = """
    query
    """
    
    con, cursor = clnjdb7()  # Establish database connection
    
    try:
        cursor.execute(query)  # Execute DELETE statement
        con.commit()  # Commit changes to the database
        print("Data deleted successfully.")
    except Exception as e:
        print(f"Error deleting data: {e}")
    finally:
        db_disconnect(con, cursor)  # Close connection


In [61]:
def data_dump(df):
    try:
        engine = db.create_engine('mssql+pymssql://user:pass@ip/db')
    except Exception as e:
        print(e,'Unable to connect to DB')

    try:
        with engine.begin() as conn:
            df.to_sql('table', conn, if_exists='append', index=False)
            print('Total Data Dumped..:', df.shape[0])
    except Exception as e:
        print('Error in data Insertion:', e)

In [62]:
delete_data()

Data deleted successfully.


In [63]:
# filtered_merg_new_row.drop(columns={'is_well_visit','is_vaccine'}, inplace=True)

In [65]:
# filtered_merg_new_row.drop(columns=['ENTERED_FROM'], inplace=True)

In [66]:
# filtered_merg_new_row.drop(columns=['LAYER_INFO','ENTERED_FROM','ID'], inplace=True)

In [67]:
filtered_merg_new_row.CLAIM_NO.nunique()

4438

In [68]:
data_dump(filtered_merg_new_row)

Total Data Dumped..: 14184


In [69]:
filtered_merg_new_row.shape[0]

14184

In [70]:
print("Calling Data Freezing Function..!")
data_freeze()

Calling Data Freezing Function..!
Data Freezed Successfully...!


0

# END